# Example: Primal Solution of the Apple versus Orange Problem
This example will familiarize students with the [linear programming primal problem](https://en.wikipedia.org/wiki/Linear_programming). We'll compute the optimal solution to a simple constrained optimization problem using Julia's optimization ecosystem, including the VLDataScienceMachineLearningPackage and JuMP with the GLPK solver.


> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Formulate a budget-constrained choice as a linear program:__ Define the fruit quantities as decision variables, write a linear utility objective from the marginal utilities, and express the budget and nonnegativity constraints. Set a finite upper bound on each quantity from the budget and that fruit's price.
> * __Predict the optimal purchase from utility per dollar:__ Compare the marginal utility per dollar of each fruit to decide whether we buy only apples, only oranges, or any mixture that spends the full budget. Explain the three cases through the slopes of the budget line and the lines of constant utility.
> * __Solve the program and read the result:__ Build the problem model from the objective coefficients, the price row, the budget, and the bounds, then solve it with the course package. Read the optimal quantities, objective value, and solver status from the solution dictionary and relate them to the predicted case.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Constants and Problem Data
Let's set the marginal utilities, $u_{i}$, the prices, $p_{i}$, and the budget for our `Apple` versus `Orange` problem. We'll store the utility coefficients in the `u` array and the prices in the `p` array.
* The $u_{i}$ coefficients (because we have a linear utility function) are the [marginal utilities](https://en.wikipedia.org/wiki/Marginal_utility), i.e., they tell us the satisfaction we gain from consuming an additional unit of good $i$. They have units of `utils/qty`
* The $p_{i}$ coefficients represent the unit cost of each good, e.g., the cost of a single apple or orange. The $p_{i}$ coefficients have units of `USD/qty.`
* Finally, the `total_budget` variable holds the amount of money we spend on apples and oranges. The `total_budget` has units of `USD`.

Three utility settings are provided; the two commented lines give the other two cases. Let's set the problem data:


In [2]:
u = [0.55, 0.45]; # coefficients for case A (apples only)
# u = [0.15, 0.55]; # coefficients for case B (oranges only)
# u = [2.0, 4.0]; # coefficients for case C (both)
p = [2.0 4.0]; # unit price of an Apple and an Orange
total_budget = 100.0; # total budget that we can spend

___

## What should we expect?

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-ThreeCases-LP-Schematic.svg" width="1000" style="display:block; margin:0 auto;" alt="Three schematic allocation cases: an optimal apple corner when the utility line is steeper than the budget line, an optimal orange corner when it is flatter, and an entire optimal budget edge when the slopes are equal.">
</p>

This type of problem will typically result in a corner solution, i.e., we will either buy only apples or only oranges (which is the type of solution we will see below). In a __very special case__, we may be indifferent between the two fruits and will buy a combination of both (in fact, there will be an infinite number of combinations that will yield the same utility if we can buy fractional amounts of fruit).

The fruit we buy will depend on the marginal utility __per dollar__, $u_{i}/p_{i}$, not on the marginal utility alone. If apples give more utility per dollar than oranges, we will buy only apples. If oranges give more utility per dollar than apples, we will buy only oranges. If the two ratios are equal (the special case), we can buy __any combination__ of apples and oranges that spends the entire budget.

> __Three solution cases:__
>
> * __Case 1:__ If the absolute value of the slope of the budget line $|m_{I}|$ __is less than__ the absolute value of the slope of the indifference curve (line of constant satisfaction, i.e., the objective function) $|m_{O}|$, we will buy only apples (the good on the x-axis).
> * __Case 2:__ If the absolute value of the slope of the budget line $|m_{I}|$ __is greater than__ the absolute value of the slope of the indifference curve (line of constant satisfaction, i.e., the objective function) $|m_{O}|$, we will buy only oranges (the good on the y-axis).
> * __Case 3 (special case):__ If the absolute value of the slope of the budget line $|m_{I}|$ __is equal to__ the absolute value of the slope of the indifference curve (line of constant satisfaction, i.e., the objective function) $|m_{O}|$, we will be indifferent between the two fruits and can buy any combination of apples and oranges that spends the entire budget, including either corner. In this case, there will be an infinite number of solutions that yield the same utility.

So what can we conclude from this analysis? A linear program will have either a corner solution (buy only one fruit) or an infinite number of solutions (the special case). Which case we see depends on the marginal utilities relative to the prices, i.e., the objective function and the constraints together. The same geometry decides how a bacterium allocates its enzymes among competing sugars; see the application note in the [lecture](CHEME-5800-L5c-Lecture-LinearProgramming-Fall-2026.ipynb).
___

## Compute the primal solution to the apple versus orange problem
In this task, we solve the `primal` linear programming problem for the unknown values in our problem, i.e., the number of apples or oranges we should purchase to maximize our happiness function. The problem we are solving is a linear programming problem of the form:

> __Fruit choice problem (linear)__
> 
> Let's choose between two fruits, `apples` and `oranges`. We have a fixed budget to spend on these fruits. The price of an apple and the price of an orange are specified above. We want to maximize our happiness (utility) from consuming these fruits, given our budget constraint.
> 
> $$
\begin{align*}
\text{maximize}~\mathcal{O}(\mathbf{x}) &= U\left(x_{1},\dots,x_{n}\right) \\
\text{subject to}~\sum_{i\in{1,\dotsc,n}}p_{i}\;{x}_{i} & \leq I\\
\text{and}~x_{i}&\geq{0}\qquad{i=1,2,\dots,n}
\end{align*}
$$
> 
> The $p_{i}\geq{0}~\forall{i}$ denotes the cost of object $i$, $x_{i}\geq{0}$ represents 
the amount of object $i$ purchased or consumed by the agent, and $I$ represents the budget that we have to spend. In this case, we'll use a __linear__ utility function of the form:
> $$
U(x) = u_{1}\cdot{x}_{1}+u_{2}\cdot{x}_{2}
$$
> where $u_{1}$ are the marginal utilities (units: `utils/qty`), $x_{1}$ denotes the quantity of `apples = 1`, and $x_{2}$ represents the number of `oranges = 2` that we purchase.

To solve this problem, let's first specify the bounds of the variables in the bounds variable. The first column of the array corresponds to the lower bound, while the second column corresponds to the upper bound for the unknown variable(s) $x_{i}$.

> __Minimum and maximum values for the unknowns__
> 
> The minimum value for the unknowns $x_{i}$ is zero, i.e., we cannot purchase a negative amount of fruit. It would be tempting to set the upper bound to infinity; however, let's think about that choice for a moment. If we have an infinite budget and an unlimited supply, we could purchase an infinite amount of fruit. However, we have a limited budget. The maximum amount we could buy would be to spend our entire budget on one fruit. Thus, the upper bound for each unknown $x_{i}$ is $x_{j} = I/p_{i}$. 

Let's specify the bounds for our problem in the `bounds::Array{Float64,2}` variable.

In [3]:
bounds = let 
    
    # initialize -
    number_of_unknowns = 2; # we have two fruits to choose from
    bounds = zeros(number_of_unknowns, 2); # initialize the bounds
    I = total_budget; # shorthand for the budget

    # set the bounds for each unknown
    for i ∈ 1:number_of_unknowns
        bounds[i, 1] = 0.0; # lower bound is 0
        bounds[i, 2] = I / p[i]; # upper bound is I/p[i]
    end

    bounds; # return
end

2×2 Matrix{Float64}:
 0.0  50.0
 0.0  25.0

Next, we create an instance of [the `MyLinearProgrammingProblemModel` model](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyLinearProgrammingProblemModel) using [the `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build) and store this model in the `primal_problem` variable. 
This model holds the data associated with the problem, e.g., the unit costs, the marginal utilities, the right-hand side vector (i.e., the budget), and the problem bounds.

In [4]:
primal_problem = build(MyLinearProgrammingProblemModel, (
    
    c = u, # coefficients in the Utility function (objective)
    A = p, # unit prices of x1 and x2 (we need this as a matrix in this formulation)
    b = [total_budget], # budget is the right-hand side
    
    # how much of x₁ and x₂ can we buy?
    lb = bounds[:,1], # lower bound
    ub = bounds[:,2] # upper bound
));

Finally, we pass the `primal_problem` variable to [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.solve), which constructs the linear program using the [JuMP domain-specific language](https://jump.dev/). 

The implementation of [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.solve) takes the data from the `primal_problem` instance, builds the various problem structures, and returns the solution as a dictionary.

> __Why the try-catch block?__ [The `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.solve) uses the [GLPK solver](https://en.wikipedia.org/wiki/GLPK) to solve the linear program. If the solver fails, [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.solve) will throw an error. To prevent the notebook from crashing, we wrap the call to [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.solve) in a `try-catch` block. If the solver fails, we catch the error and print a message to the user.

Let's solve the primal problem and store the result in the `primal_solution_dictionary::Dict{String,Any}` variable:

In [5]:
primal_solution_dictionary = let

    # initialize -
    primal_solution = nothing;
    try
        primal_solution = solve(primal_problem) # call the solver
    catch error
        println(error)
    end
    
    primal_solution; # return
end;

What's in the solution dictionary?

In [6]:
primal_solution_dictionary

Dict{String, Any} with 3 entries:
  "argmax"          => [50.0, 0.0]
  "status"          => OPTIMAL
  "objective_value" => 27.5

The `argmax` key points to the optimal values of the unknowns, i.e., the number of apples and oranges we should purchase to maximize our happiness function. The `objective_value` key points to the optimal value of the objective function, i.e., the maximum happiness we can achieve given our budget constraint. The `status` key holds the status of the optimization problem, e.g., whether it was solved successfully or not.

> __Note__: In the implementation of [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.solve) we have an @assert statement that checks that the solver reports a solved and feasible result. If it does not, the @assert statement will throw an error. This is a safety check so that a returned dictionary always describes a successfully solved problem.

So, which fruit should we buy (and how much of each)? It depends on the coefficients in the utility function (the marginal utilities). Try changing the values in the `u` variable and re-running the notebook to see how the solution changes.

In [7]:
let

    # initialize -
    df = DataFrame();
    labels = ["apples", "oranges"];
    x_opt = primal_solution_dictionary["argmax"]; # get the optimal values of the unknowns
    number_of_unknowns = length(x_opt); # number of unknowns

    # populate the dataframe -
    for i ∈ 1:number_of_unknowns
        row_df = (
            fruit = labels[i],
            quantity = x_opt[i]
        )
        push!(df, row_df)
    end

    pretty_table(df, backend = :text,
         table_format = TextTableFormat(borders = text_table_borders__compact))
end

 --------- ----------
    fruit   quantity 
   String    Float64 
 --------- ----------
   apples       50.0
  oranges        0.0
 --------- ----------


___

## Summary
In this example, we formulated the apple versus orange purchase as a linear program, predicted its solution from utility per dollar, and solved it with the course package to compare the result with the prediction.

> __Key Takeaways:__
>
> * __Budget allocation with linear utility gives corner solutions:__ We typically spend the entire budget on one fruit, and the fruit with the higher marginal utility per dollar wins. Only when the two utility-per-dollar ratios are equal does every full-budget mixture become optimal.
> * __Slopes predict the solution before we solve:__ We compared the slope of the budget line with the slope of the constant-utility lines to identify which corner, or which edge, the optimum lands on. The three schematic cases correspond to the three utility settings in the data cell.
> * __Setting up the model is the real work:__ We built the problem from the utility coefficients, the price row, the budget, and finite bounds derived from the budget, then read the optimal quantities, objective value, and status from the solution dictionary. The solver mechanics stayed inside the course package.

Try changing the `u` coefficients in the second code cell and re-running the notebook to see these principles in action!
